In [0]:
from pyspark.sql.functions import col, sha2, concat_ws, lit

# 1. Odczytujemy nowe dane z warstwy Bronze
bronze_hr_df = spark.read.table("dbw_showcase.default.hr_bronze")

# 2. Generujemy Surrogate Key (SHA-2) - Wymóg z rozmowy rekrutacyjnej!
# Łączymy ID pracownika i czas aktualizacji, a następnie to haszujemy.
silver_hr_stg = bronze_hr_df.withColumn(
    "emp_sk", sha2(concat_ws("-", col("emp_id"), col("updated_at")), 256)
)

# Tworzymy tymczasowy widok, aby użyć go w zapytaniu SQL MERGE
silver_hr_stg.createOrReplaceTempView("stg_hr_updates")

# 3. Inicjalizacja tabeli Silver (uruchomi się tylko za pierwszym razem, tworząc strukturę)
spark.sql("""
CREATE TABLE IF NOT EXISTS dbw_showcase.default.hr_silver (
    emp_sk STRING,
    emp_id INT,
    emp_name STRING,
    city STRING,
    dept_id INT,
    department_name STRING,
    valid_from TIMESTAMP,
    valid_to TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA
""")

# 4. Magia Databricks: MERGE INTO dla SCD Type 2
# Ten kod sprawdza, czy pracownik już istnieje. Jeśli się zmienił, zamyka stary rekord i dodaje nowy.
print("⏳ Uruchamiam proces MERGE INTO (SCD Type 2)...")

spark.sql("""
MERGE INTO dbw_showcase.default.hr_silver AS target
USING (
    -- Wybieramy nowe rekordy z Bronze i przygotowujemy je do logiki SCD2
    SELECT 
        emp_sk, emp_id, emp_name, city, dept_id, department_name, 
        CAST(updated_at AS TIMESTAMP) AS valid_from 
    FROM stg_hr_updates
) AS source
ON target.emp_id = source.emp_id AND target.is_current = true

-- Kiedy pracownik istnieje, ale zmieniły się jego dane (np. przeprowadził się):
WHEN MATCHED AND (target.city <> source.city OR target.department_name <> source.department_name) THEN
  UPDATE SET 
    target.is_current = false, 
    target.valid_to = source.valid_from

-- Kiedy to zupełnie nowy pracownik (lub dokładamy nowy rekord dla zaktualizowanego pracownika):
WHEN NOT MATCHED THEN
  INSERT (emp_sk, emp_id, emp_name, city, dept_id, department_name, valid_from, valid_to, is_current)
  VALUES (source.emp_sk, source.emp_id, source.emp_name, source.city, source.dept_id, source.department_name, source.valid_from, null, true)
""")

print("✅ Tabela Silver HR została zaktualizowana zgodnie z logiką SCD Type 2!")

In [0]:
# ==========================================
# SYMULACJA: "DZIEŃ 2" - PRACOWNIK ZMIENIA MIASTO
# ==========================================

print("1. Bierzemy jednego pracownika i przenosimy go do Warszawy (symulacja nowych danych w Bronze)...")
spark.sql("""
    INSERT INTO dbw_showcase.default.hr_bronze
    SELECT 
        dept_id, 
        department_name, 
        emp_id, 
        emp_name, 
        'Warszawa (Nowa Centrala)' AS city, -- ZMIANA MIASTA!
        current_timestamp() AS updated_at, 
        current_timestamp() AS ingested_at
    FROM dbw_showcase.default.hr_bronze
    LIMIT 1
""")

print("2. Uruchamiamy ponownie nasz rurociąg MERGE INTO (SCD Type 2)...")
spark.sql("""
    MERGE INTO dbw_showcase.default.hr_silver AS target
    USING (
        -- NAPRAWA: Generujemy emp_sk (SHA-2) w locie dla nowych danych
        SELECT 
            sha2(concat_ws('-', emp_id, updated_at), 256) AS emp_sk, 
            emp_id, 
            emp_name, 
            city, 
            dept_id, 
            department_name, 
            CAST(updated_at AS TIMESTAMP) AS valid_from 
        FROM (
            -- Bierzemy najświeższy rekord z Bronze dla każdego pracownika
            SELECT *, ROW_NUMBER() OVER(PARTITION BY emp_id ORDER BY updated_at DESC) as rn 
            FROM dbw_showcase.default.hr_bronze
        ) WHERE rn = 1
    ) AS source
    ON target.emp_id = source.emp_id AND target.is_current = true

    WHEN MATCHED AND (target.city <> source.city OR target.department_name <> source.department_name) THEN
      UPDATE SET target.is_current = false, target.valid_to = source.valid_from

    WHEN NOT MATCHED THEN
      INSERT (emp_sk, emp_id, emp_name, city, dept_id, department_name, valid_from, valid_to, is_current)
      VALUES (source.emp_sk, source.emp_id, source.emp_name, source.city, source.dept_id, source.department_name, source.valid_from, null, true)
""")

print("3. Wynik! Spójrz na tabelę poniżej. Zobaczysz DWIE wersje tego samego pracownika!")
display(
    spark.sql("""
        SELECT emp_id, emp_name, city, is_current, valid_from, valid_to 
        FROM dbw_showcase.default.hr_silver 
        WHERE emp_id = (SELECT emp_id FROM dbw_showcase.default.hr_bronze WHERE city = 'Warszawa (Nowa Centrala)' LIMIT 1)
        ORDER BY valid_from DESC
    """)
)